# POIs del Corredor — Av. Roosevelt · Overture Maps
**ITT Cali Inteligente · Equipo de Gobierno de Datos**

Descarga todos los puntos de interés (negocios, servicios, equipamiento) dentro del polígono de intervención de Av. Roosevelt usando Overture Maps — dataset open data de Meta, Microsoft, Amazon y TomTom.

**Sistema de referencia:** WGS84 (EPSG:4326) para toda la información geoespacial.

**Datos:** Se cargan automáticamente desde el repositorio Git del proyecto.

**Instrucciones:**
1. Ejecutar las celdas en orden
2. Los datos se cargan automáticamente desde Git
3. Los resultados y mapas se generan automáticamente

## Celda 1 — Instalación de dependencias

In [ ]:
import subprocess, sys
def check_pkg(pkg):
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for p in ['overturemaps', 'geopandas', 'matplotlib', 'pandas', 'numpy', 'folium', 'shapely']:
    check_pkg(p)
print('Dependencias verificadas')

## Celda 2 — Importaciones y configuración

In [ ]:
import os, json, warnings, datetime
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from shapely.geometry import Point
warnings.filterwarnings('ignore')

# Sistema de referencia
CRS_WGS84 = 'EPSG:4326'

plt.rcParams.update({
    'figure.facecolor': '#F4F6F9',
    'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3
})
print(f'Sistema de referencia de trabajo: {CRS_WGS84}')
print('Configuración lista')

## Celda 3 — Parámetros, rutas y detección de entorno

In [ ]:
# Repositorio del proyecto (datos del corredor)
REPO_URL = 'https://github.com/j0rg3c45/indice-caminabilidad-roosevelt.git'
REPO_NAME = 'indice-caminabilidad-roosevelt'

# Repositorio de este proyecto (Índice de Concurrencia)
REPO_CONCURRENCIA_URL = 'https://github.com/j0rg3c45/Indice_Concurrencia.git'
REPO_CONCURRENCIA_NAME = 'Indice_Concurrencia'

# Detectar entorno: Colab o local
if os.path.exists('/content'):
    import subprocess as _sp
    os.chdir('/content')
    # Clonar repo de datos del corredor
    _sp.getoutput(f'rm -rf {REPO_NAME}')
    print(_sp.getoutput(f'git clone {REPO_URL}'))
    PROJECT_ROOT = Path(f'/content/{REPO_NAME}')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent

# Rutas
DATA_DIR = PROJECT_ROOT / 'data' / 'itt_roosevelt' / 'Roosevelt' / 'Geojson_Roosevelt'
OUTPUT_DIR = Path(os.getcwd()).parent / 'outputs' if not os.path.exists('/content') else Path('/content') / REPO_CONCURRENCIA_NAME / 'outputs'
DATA_OUT_DIR = Path(os.getcwd()).parent / 'data' if not os.path.exists('/content') else Path('/content') / REPO_CONCURRENCIA_NAME / 'data'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Buscar polígono del buffer
import re
def find_file(base_dir, pattern):
    regex = re.compile(pattern, re.IGNORECASE)
    for root, _dirs, files in os.walk(base_dir):
        for f in files:
            if regex.search(f) and f.lower().endswith('.geojson'):
                return os.path.join(root, f)
    return None

POLIGONO_PATH = find_file(DATA_DIR, r'Geojson_tramos_Roosevelt_Buffer_100')

ZONA_NOMBRE = 'Av. Roosevelt — Cali'
POLIGONO_NOMBRE = 'Av. Roosevelt - Buffer 100m'

print(f'DATA_DIR: {DATA_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'Polígono: {"✅ " + Path(POLIGONO_PATH).name if POLIGONO_PATH else "❌ NO ENCONTRADO"}')

## Celda 4 — Cargar polígono y definir bounding box

In [ ]:
# Cargar polígono de intervención (buffer 100m)
gdf_poligono = gpd.read_file(POLIGONO_PATH)

# Asegurar WGS84
if gdf_poligono.crs is None:
    gdf_poligono = gdf_poligono.set_crs(CRS_WGS84)
elif gdf_poligono.crs.to_epsg() != 4326:
    gdf_poligono = gdf_poligono.to_crs(CRS_WGS84)

poligono = gdf_poligono.geometry.iloc[0]

# Bounding box para descarga de Overture Maps
bounds = gdf_poligono.total_bounds  # xmin, ymin, xmax, ymax
BBOX = (bounds[0], bounds[1], bounds[2], bounds[3])

print(f'\n=== ÁREA DE INTERVENCIÓN ===')
print(f'  Zona: {ZONA_NOMBRE}')
print(f'  CRS: {gdf_poligono.crs}')
print(f'  Bounding box: {BBOX}')
print(f'  Polígono cargado desde repositorio Git')

## Celda 5 — Descargar POIs desde Overture Maps

In [ ]:
import subprocess

xmin, ymin, xmax, ymax = BBOX
RAW_FILE = 'roosevelt_overture_poi_raw.geojson'

cmd = [
    'overturemaps', 'download',
    f'--bbox={xmin},{ymin},{xmax},{ymax}',
    '-f', 'geojson',
    '--type=place',
    '-o', RAW_FILE
]

print('Descargando POIs de Overture Maps...')
print(f'Área: {POLIGONO_NOMBRE}')
print(f'Bounding box: {BBOX}')
print()

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    size = os.path.getsize(RAW_FILE) / 1024
    print(f'✅ Descarga exitosa — archivo: {size:.1f} KB')
else:
    print(f'❌ Error: {result.stderr}')

## Celda 6 — Filtrar POIs al polígono exacto

In [ ]:
# Cargar POIs descargados
gdf = gpd.read_file(RAW_FILE)

# Asegurar CRS
if gdf.crs is None:
    gdf = gdf.set_crs(CRS_WGS84)

# Filtrar por polígono exacto
gdf_filtrado = gdf[gdf.geometry.within(poligono)].copy()

print(f'=== FILTRADO ESPACIAL ===')
print(f'  POIs en bounding box: {len(gdf)}')
print(f'  POIs dentro del polígono exacto: {len(gdf_filtrado)}')
print(f'  POIs descartados: {len(gdf) - len(gdf_filtrado)}')

## Celda 7 — Extraer atributos y clasificar en macrocategorías

In [ ]:
# Funciones de extracción
def extraer_nombre(row):
    try:
        names = row.get('names', {})
        if isinstance(names, str):
            names = json.loads(names)
        primary = names.get('primary', '')
        if isinstance(primary, list):
            return primary[0] if primary else 'Sin nombre'
        return primary or 'Sin nombre'
    except:
        return 'Sin nombre'

def extraer_categoria(row):
    try:
        cats = row.get('categories', {})
        if isinstance(cats, str):
            cats = json.loads(cats)
        return cats.get('primary', 'sin_categoria')
    except:
        return 'sin_categoria'

def extraer_confianza(row):
    try:
        return round(float(row.get('confidence', 0)), 2)
    except:
        return None

# Agrupación en macrocategorías
def agrupar(cat):
    cat = str(cat).lower()
    if any(x in cat for x in ['restaurant','food','bakery','pizza','burger',
                                'chicken','coffee','bar','dessert','ice_cream',
                                'colombian','peruvian','vegetarian','fast_food','cafe','eat_and_drink']):
        return 'Gastronomía y bebidas'
    if any(x in cat for x in ['health','medical','dental','doctor','hospital',
                                'clinic','pharmacy','optician','nursing',
                                'counseling','mental_health','holistic','veterinarian']):
        return 'Salud y bienestar'
    if any(x in cat for x in ['store','shop','supermarket','market','clothing',
                                'electronics','furniture','sporting','jewelry',
                                'beauty_supply','thrift','wholesale','department',
                                'home_goods','candy','grocery','flowers']):
        return 'Comercio y tiendas'
    if any(x in cat for x in ['school','university','college','education',
                                'language','driving_school','dance','music',
                                'yoga','martial_arts','nursing_school','cosmetology','art_school']):
        return 'Educación y formación'
    if any(x in cat for x in ['hotel','lodging','hostel','accommodation']):
        return 'Alojamiento'
    if any(x in cat for x in ['professional','financial','insurance','accounting',
                                'marketing','advertising','consulting','technology',
                                'media','b2b','printing','business','legal','lawyer','software']):
        return 'Servicios profesionales'
    if any(x in cat for x in ['automotive','car','motorcycle','tire','bike',
                                'gas_station','transportation']):
        return 'Automotriz y transporte'
    if any(x in cat for x in ['beauty','spa','salon','barber','hair',
                                'esthetic','cosmetic']):
        return 'Belleza y estética'
    if any(x in cat for x in ['sport','gym','fitness','recreation','basketball',
                                'stadium','arena','active','park']):
        return 'Deporte y recreación'
    if any(x in cat for x in ['church','religious','cathedral','catholic']):
        return 'Religioso'
    if any(x in cat for x in ['community','social','nonprofit','organization',
                                'government','public','association','foundation']):
        return 'Comunitario e institucional'
    if any(x in cat for x in ['art','museum','landmark','historical','event',
                                'concert','entertainment','adult','travel']):
        return 'Cultura y entretenimiento'
    return 'Otros'

# Construir DataFrame limpio
df_lista = pd.DataFrame({
    'nombre': gdf_filtrado.apply(extraer_nombre, axis=1),
    'categoria': gdf_filtrado.apply(extraer_categoria, axis=1),
    'confianza': gdf_filtrado.apply(extraer_confianza, axis=1),
    'lat': gdf_filtrado.geometry.y,
    'lon': gdf_filtrado.geometry.x,
})
df_lista['macrocategoria'] = df_lista['categoria'].apply(agrupar)

print(f'=== POIs PROCESADOS ===')
print(f'Total: {len(df_lista)}')
print()
print('Distribución por macrocategoría:')
print(df_lista['macrocategoria'].value_counts().to_string())

## Celda 8 — Mapa interactivo con capas (Folium)

In [ ]:
# Paleta de colores por macrocategoría
PALETTE = {
    'Gastronomía y bebidas':       '#E74C3C',
    'Salud y bienestar':           '#3498DB',
    'Comercio y tiendas':          '#F39C12',
    'Educación y formación':       '#9B59B6',
    'Alojamiento':                 '#1ABC9C',
    'Servicios profesionales':     '#2C3E50',
    'Automotriz y transporte':     '#7F8C8D',
    'Belleza y estética':          '#E91E8C',
    'Deporte y recreación':        '#27AE60',
    'Religioso':                   '#8E44AD',
    'Comunitario e institucional': '#16A085',
    'Cultura y entretenimiento':   '#D35400',
    'Otros':                       '#BDC3C7',
}

# Mapa interactivo — todo en WGS84
centroid = poligono.centroid
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=15, tiles=None)

# Capas base
folium.TileLayer('CartoDB positron', name='CartoDB Claro').add_to(m)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri', name='Esri Satélite', overlay=False
).add_to(m)

# Google tiles
google_streets = 'http://{s}.google.com/vt/lyrs=m&x={x}&y={y}&z={z}'
google_satellite = 'http://{s}.google.com/vt/lyrs=s&x={x}&y={y}&z={z}'
folium.TileLayer(
    tiles=google_streets,
    attr='Google', name='Google Streets',
    subdomains=['mt0', 'mt1', 'mt2', 'mt3'], overlay=False
).add_to(m)
folium.TileLayer(
    tiles=google_satellite,
    attr='Google', name='Google Satélite',
    subdomains=['mt0', 'mt1', 'mt2', 'mt3'], overlay=False
).add_to(m)

# Polígono de intervención
folium.GeoJson(
    gdf_poligono.__geo_interface__,
    name='Polígono intervención (Buffer 100m)',
    style_function=lambda x: {'color': '#1B4F8A', 'fillColor': '#2E7D32',
                              'fillOpacity': 0.08, 'weight': 2}
).add_to(m)

# POIs por macrocategoría como capas separadas
for mac, grupo in df_lista.groupby('macrocategoria'):
    color = PALETTE.get(mac, '#BDC3C7')
    fg = folium.FeatureGroup(name=f'{mac} ({len(grupo)})', show=True)
    for _, row in grupo.iterrows():
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color=color,
            fill=True,
            fill_opacity=0.7,
            weight=1,
            popup=folium.Popup(
                f"<b>{row['nombre']}</b><br>"
                f"Categoría: {row['categoria']}<br>"
                f"Macrocategoría: {mac}<br>"
                f"Confianza: {row['confianza']}",
                max_width=250
            )
        ).add_to(fg)
    fg.add_to(m)

folium.LayerControl().add_to(m)
print(f'Mapa interactivo generado — {len(df_lista)} POIs en {len(PALETTE)} macrocategorías')
print(f'CRS: {CRS_WGS84}')
m

## Celda 9 — Mapa estático con categorías agrupadas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 14))

# Polígono
gdf_poligono.plot(
    ax=ax, color='#CB4335', alpha=0.10,
    edgecolor='#CB4335', linewidth=1.5
)

# POIs por macrocategoría
for mac, grupo in df_lista.groupby('macrocategoria'):
    ax.scatter(
        grupo['lon'], grupo['lat'],
        color=PALETTE.get(mac, '#BDC3C7'),
        s=18, zorder=5, alpha=0.85,
        edgecolors='white', linewidth=0.3,
        label=f"{mac} ({len(grupo)})"
    )

ax.set_title(
    f'POIs Av. Roosevelt — Overture Maps\n'
    f'{len(df_lista)} lugares · Línea base {datetime.date.today().strftime("%B %Y")}',
    fontsize=12, pad=12
)
ax.set_xlabel('Longitud', fontsize=9)
ax.set_ylabel('Latitud', fontsize=9)
ax.tick_params(labelsize=8)

ax.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    fontsize=9,
    framealpha=0.9,
    borderpad=0.8,
    handlelength=1.2,
    title='Categoría (n lugares)',
    title_fontsize=9
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roosevelt_overture_poi_mapa.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mapa guardado: {OUTPUT_DIR / "roosevelt_overture_poi_mapa.png"}')

## Celda 10 — Exportar CSV limpio para el datalake

In [ ]:
df_export = df_lista.copy()
df_export['fecha_extraccion'] = datetime.date.today().isoformat()
df_export['poligono'] = POLIGONO_NOMBRE
df_export['fuente'] = 'Overture Maps Foundation'
df_export['momento'] = 'linea_base_pre_intervencion'
df_export['release_overture'] = '2026-04-15'

# Exportar
csv_path = DATA_OUT_DIR / 'roosevelt_overture_poi.csv'
geojson_path = DATA_OUT_DIR / 'roosevelt_overture_poi.geojson'

df_export.to_csv(csv_path, index=False)
gdf_filtrado.to_file(geojson_path, driver='GeoJSON')

print(f'=== EXPORTACIÓN ===')
print(f'  CSV: {csv_path.name} ({len(df_export)} registros)')
print(f'  GeoJSON: {geojson_path.name}')
print()
print('Columnas del CSV:')
for col in df_export.columns:
    print(f'  - {col}')
print()
print('Para comparar después de las obras: cambiar momento a post_intervencion')
print('y repetir con el release más reciente de Overture Maps.')